# CRISP-DM: Donor Impact Allocation

**Analytics goal:** Estimate **which program areas** donations tend to support and **what share** flows to priorities (e.g. education)—to power **transparent donor impact** storytelling and internal targeting.

| CRISP-DM phase | Where it appears |
|----------------|------------------|
| **1. Business understanding** | Next sections (problem framing, KPI mapping) |
| **2. Data understanding** | Donor/donation snapshot, audit |
| **3. Data preparation** | `src.features` |
| **4. Modeling** | Top-area classifier + share explanation |
| **5. Evaluation** | Metrics + business interpretation |
| **6. Deployment** | Donor profile / impact panel |

### 1) Business understanding (impact & allocation narrative)

**Organizational context:** Donors increasingly ask *“where does my money go?”* Lighthouse Sanctuary must connect gifts to **mission programs** (education, safe housing, reintegration, etc.) in a way that is **honest**, **understandable**, and **aligned with fundraising messaging**—without implying false precision.

**Business objectives**
- Provide **credible, donor-facing estimates** of programmatic emphasis (e.g. likely top allocation area, education share) for stewardship and campaigns.
- Help **donor relations** tailor stories: which impact themes resonate for similar past gifts.

**Stakeholders and decisions**
| Stakeholder | Decision enabled |
|-------------|------------------|
| **Donor relations / communications** | Impact copy, annual report callouts, personalized summaries |
| **Fundraising ops** | Segmenting donors by inferred interest themes (for testing, not stereotyping) |
| **Leadership** | Aggregate narrative on program funding mix (with aggregation, not individual overclaim) |

**Measurable success criteria (model + program)**
- Model: reasonable **top-program classification** quality and **share fit** (e.g. macro-F1, R²) as in KPI mapping; non-trivial lift vs. baseline.
- Program: **consistent, stable** estimates across periods where business expects stability; **guardrails** on overclaiming “your gift built X” when uncertainty is high.

**Constraints**
- **Not causal proof:** Historical allocation patterns reflect **accounting and operations**, not counterfactual “your dollar caused exactly this outcome.”
- **Ethics & trust:** Avoid manipulative precision; offer ranges or qualitative bands where appropriate.
- **Equity:** Messaging should not stereotype donors; use segments responsibly.

**Costs of errors**
- Wrong **top program** → misleading thank-you or campaign creative.
- Wrong **share** → mis-set donor expectations and reputational risk.

**Use of outputs:** **Narrative support** and internal triage—paired with finance/program verification for major public claims.

In [1]:
import os, sys
from pathlib import Path


def _find_ml_pipelines_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        mp = base / "ml-pipelines"
        if mp.is_dir() and (mp / "requirements.txt").is_file():
            return mp
        if base.name == "ml-pipelines" and (base / "requirements.txt").is_file():
            return base
    raise RuntimeError(
        "Could not find ml-pipelines/requirements.txt. Open the Lighthouse-Sanctuary repo "
        "or the ml-pipelines folder, then use Run All."
    )


_boot_path = _find_ml_pipelines_root() / "pipeline_common" / "notebook_bootstrap.py"
_ns = globals()
_ns["__file__"] = str(_boot_path)
exec(compile(_boot_path.read_text(encoding="utf-8"), str(_boot_path), "exec"), _ns)

import pandas as pd

from src.db import load_env, build_engine
from src.features import build_frame, split_xy
from src.modeling import train_predictive_top_area, train_explanatory_share, donor_impact_output


## Problem Framing
- Predictive objective: predict top allocation program area for a donation.
- Explanatory objective: estimate which features are associated with higher education allocation share.
- Stakeholder: donor relations and fundraising operations teams.

## Business KPI mapping (operational measures)

Quantifies the **business objectives** in Business understanding for model and program review.

- Decision enabled: donor impact estimate display and targeting narrative.
- Primary KPI: top-program prediction quality (macro F1) and allocation-share fit (R2).
- Guardrails: avoid overclaiming causal outcomes; maintain estimate stability across periods.
- Success criteria: non-trivial predictive lift and interpretable driver consistency.

In [2]:
load_env('.env')
engine = build_engine(os.getenv('DB_PROFILE', 'local'))
df = build_frame(engine)
X, y_top, y_share_education = split_xy(df)
print('Rows:', len(df), '| Distinct donors:', df['supporter_id'].nunique())
df.head()

Rows: 420 | Distinct donors: 59


,donation_id,supporter_id,donation_date,donation_value,is_recurring,channel_source,donation_type,campaign_name,supporter_type,relationship_type,...,alloc_maintenance,alloc_outreach,total_allocated,share_education,share_wellbeing,share_operations,share_transport,share_maintenance,share_outreach,top_program_area
0,1,42,2025-12-31,717.179993,False,Campaign,Monetary,None,MonetaryDonor,Local,...,0.0,0.0,717.179993,1.0,0.0,0.0,0.0,0.0,0.0,education
1,2,25,2025-12-02,35.150002,True,Event,Time,Year-End Hope,MonetaryDonor,Local,...,0.0,0.0,35.150002,0.0,0.0,0.0,1.0,0.0,0.0,transport
2,3,19,2024-12-02,1074.650024,False,PartnerReferral,Monetary,None,Volunteer,Local,...,0.0,0.0,1074.650024,0.0,1.0,0.0,0.0,0.0,0.0,wellbeing
3,4,33,2023-09-11,1230.560059,False,PartnerReferral,Monetary,None,InKindDonor,Local,...,0.0,0.0,799.859985,0.0,0.0,1.0,0.0,0.0,0.0,operations
4,5,24,2023-11-08,1177.410034,True,SocialMedia,InKind,GivingTuesday,MonetaryDonor,Local,...,0.0,0.0,1177.410034,0.0,0.0,1.0,0.0,0.0,0.0,operations


In [3]:
# Data Understanding Audit
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
display(missing_pct.head(10).to_frame('missing_pct'))
audit = {
    'rows': len(df),
    'program_area_classes': int(y_top.nunique()),
    'zero_donation_value_rows': int((df['donation_value'] <= 0).sum()),
}
print('Audit summary:', audit)
print('Feature rationale: donation/supporter context explains likely allocation patterns observed in history.')

,missing_pct
donation_id,0.0
supporter_id,0.0
share_outreach,0.0
share_maintenance,0.0
share_transport,0.0
share_operations,0.0
share_wellbeing,0.0
share_education,0.0
total_allocated,0.0
alloc_outreach,0.0


Audit summary: {'rows': 420, 'program_area_classes': 6, 'zero_donation_value_rows': 0}
Feature rationale: donation/supporter context explains likely allocation patterns observed in history.


In [4]:
pred_metrics, best_top_model = train_predictive_top_area(X, y_top)
print('Predictive metrics (top program area):')
display(pred_metrics)

Predictive metrics (top program area):


,model,accuracy,macro_f1
0,rf_top_area,0.209524,0.181973
1,logistic_top_area,0.180952,0.178687


In [5]:
expl_metrics, linear_share_model, rf_share_model, coef_df = train_explanatory_share(X, y_share_education)
print('Share model metrics:')
display(expl_metrics)
print('Top positive explanatory drivers (education share):')
display(coef_df.head(12))
print('Top negative explanatory drivers:')
display(coef_df.tail(12).sort_values('coefficient'))

Share model metrics:


,model,r2,mae
0,linear_share_education,-0.108636,0.360979
1,rf_share_education,-0.167879,0.363019


Top positive explanatory drivers (education share):


,feature,coefficient
7,cat__channel_source_PartnerReferral,0.129911
14,cat__campaign_name_Back to School,0.096112
20,cat__supporter_type_MonetaryDonor,0.082874
24,cat__supporter_type_Volunteer,0.062562
12,cat__donation_type_SocialMedia,0.057743
35,cat__acquisition_channel_Website,0.052425
11,cat__donation_type_Skills,0.051707
27,cat__relationship_type_PartnerOrganization,0.050318
2,num__donation_value,0.047343
32,cat__acquisition_channel_Event,0.039512


Top negative explanatory drivers:


,feature,coefficient
22,cat__supporter_type_SkillsContributor,-0.077421
34,cat__acquisition_channel_SocialMedia,-0.076857
13,cat__donation_type_Time,-0.066912
18,cat__campaign_name_Year-End Hope,-0.058327
4,cat__channel_source_Campaign,-0.057573
25,cat__relationship_type_International,-0.054729
15,cat__campaign_name_GivingTuesday,-0.050856
29,cat__region_Mindanao,-0.046790
8,cat__channel_source_SocialMedia,-0.042932
31,cat__acquisition_channel_Church,-0.041777


In [6]:
impact_out = donor_impact_output(df, best_top_model, linear_share_model)
print('Donor impact output sample:')
display(impact_out.head(20))

Donor impact output sample:


,donation_id,supporter_id,donation_value,pred_top_program_area,pred_share_education,pred_share_education_low,pred_share_education_high
0,185,55,6481.540039,maintenance,0.671440,0.571440,0.771440
1,95,6,3189.709961,operations,0.629222,0.529222,0.729222
2,357,3,2133.989990,operations,0.593951,0.493951,0.693951
3,80,32,1997.699951,education,0.555118,0.455118,0.655118
4,346,9,1604.349976,education,0.535224,0.435224,0.635224
5,227,3,1053.819946,education,0.445381,0.345381,0.545381
6,4,33,1230.560059,operations,0.428934,0.328934,0.528934
7,188,24,1061.849976,education,0.427227,0.327227,0.527227
8,306,16,350.190002,education,0.423165,0.323165,0.523165
9,16,45,4026.840088,transport,0.422894,0.322894,0.522894


## Evaluation in Business Terms
- Misclassifying top area can mislead donor messaging.
- Over/under-estimating allocation shares can distort expectation setting.
- Use outputs as impact estimates based on historical patterns, not guaranteed causal outcomes.

## Operationalization Policy + Monitoring
- Prediction policy: display top-program estimate + education-share range.
- Action bands: high education-share estimates for education-focused messaging.
- Retraining cadence: monthly or when monitored quality drops.
- References:
  - `ml-pipelines/integration/pipeline_registry.yaml`
  - `ml-pipelines/integration/monitoring_spec.md`
  - `ml-pipelines/integration/README.md`

## Deployment Notes
Integrate as donor profile panel: input donation context, output estimated allocation profile and uncertainty range.